<a href="https://colab.research.google.com/github/Zainab-Binte-Khalid/urdu-ocr-codesaviours-si26-zainab/blob/main/SI26_Week03_Zainab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.listdir('/content/drive/MyDrive/external_datasets'))

In [ ]:
!apt-get install -y unrar -q
!unrar x "/content/drive/MyDrive/external_datasets/Cropped_word_dataset.rar" "/content/word_dataset/"

In [ ]:
import os

for root, dirs, files in os.walk('/content/word_dataset'):
    level = root.replace('/content/word_dataset', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:
        print(f'{subindent}{file}')
    if len(files) > 5:
        print(f'{subindent}... and {len(files)-5} more files')

In [ ]:
# Check the ground truth files
print('=== gt_file.tsv (first 5 lines) ===')
with open('/content/word_dataset/Cropped_word_dataset/gt_file.tsv', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.strip())

print('\n=== annotation_train.txt (first 5 lines) ===')
with open('/content/word_dataset/Cropped_word_dataset/annotation_train.txt', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.strip())

print('\n=== imlist.txt (first 5 lines) ===')
with open('/content/word_dataset/Cropped_word_dataset/imlist.txt', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.strip())

In [ ]:
with open('/content/word_dataset/Cropped_word_dataset/gt_file.tsv', 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f'Total entries: {len(lines)}')
print('\nSample of 20 entries:')
for line in lines[:20]:
    print(line.strip())

In [ ]:
import random
import os

random.seed(42)

gt_path = '/content/word_dataset/Cropped_word_dataset/gt_file.tsv'
entries = []
with open(gt_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            filename, text = parts
            # Basic quality filter: skip very short (likely fragments) or empty text
            if len(text.strip()) >= 2:
                entries.append((filename, text))

print(f'Total valid entries after basic filtering: {len(entries)}')

sample_size = min(1000, len(entries))
sampled = random.sample(entries, sample_size)

print(f'Sampled: {len(sampled)} entries')
print('\nFirst 10 samples:')
for f, t in sampled[:10]:
    print(f, '->', t)

In [ ]:
import os

source_folder = '/content/word_dataset/Cropped_word_dataset/word_dataset'
print('Folder exists:', os.path.exists(source_folder))
print('Total files in folder:', len(os.listdir(source_folder)))
print('First 5 files:', os.listdir(source_folder)[:5])

# Try opening one actual image to confirm it's a real, readable image file
from PIL import Image
sample_file = os.listdir(source_folder)[0]
img = Image.open(os.path.join(source_folder, sample_file))
print(f'\nSample image "{sample_file}" size:', img.size)

In [ ]:
import shutil
import os

source_folder = '/content/word_dataset/Cropped_word_dataset/word_dataset'
dest_folder = '/content/external_word_sample'
os.makedirs(dest_folder, exist_ok=True)

copied = 0
missing = []

for filename, text in sampled:
    src = os.path.join(source_folder, filename)
    dst = os.path.join(dest_folder, filename)
    if os.path.exists(src):
        shutil.copy(src, dst)
        copied += 1
    else:
        missing.append(filename)

print(f'Copied: {copied}')
print(f'Missing: {len(missing)}')
if missing:
    print('First few missing:', missing[:5])

In [ ]:
import shutil
import os

source_folder = '/content/word_dataset/Cropped_word_dataset/word_dataset'
dest_folder = '/content/external_word_sample'
os.makedirs(dest_folder, exist_ok=True)

copied = 0
missing = []

for filename, text in sampled:
    src = os.path.join(source_folder, filename)
    dst = os.path.join(dest_folder, filename)
    if os.path.exists(src):
        shutil.copy(src, dst)
        copied += 1
    else:
        missing.append(filename)

print(f'Copied: {copied}')
print(f'Missing: {len(missing)}')
if missing:
    print('First few missing:', missing[:5])

In [ ]:
import csv
import os

# Rename files with a clean prefix for consistency with your naming convention
final_folder = 'external_word_sample_renamed'
os.makedirs(final_folder, exist_ok=True)

import shutil
new_entries = []

for i, (filename, text) in enumerate(sampled, 1):
    src = os.path.join('/content/external_word_sample', filename)
    ext = os.path.splitext(filename)[1]
    new_name = f'scene_word_{i:04d}{ext}'
    shutil.copy(src, os.path.join(final_folder, new_name))
    new_entries.append({
        'image': f'images/scene_word/{new_name}',
        'text': text,
        'source': 'natural_scene_dataset'
    })

print(f'Renamed and organized {len(new_entries)} entries')
print('\nFirst 5 entries:')
for e in new_entries[:5]:
    print(e)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

fig, axes = plt.subplots(10, 1, figsize=(6, 25))

for i in range(10):
    entry = new_entries[i]
    filename = os.path.basename(entry['image'])
    img_path = os.path.join('external_word_sample_renamed', filename)
    img = Image.open(img_path)

    axes[i].imshow(img)
    axes[i].set_title(f'{filename}  →  "{entry["text"]}"', fontsize=12)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('scene_word_check.png', dpi=100, bbox_inches='tight')
plt.show()

from google.colab import files
files.download('scene_word_check.png')

In [ ]:
print('Raw text values (bypass matplotlib rendering):')
for i in range(10):
    filename, text = sampled[i]
    print(f'{filename}: "{text}"  (repr: {repr(text)})')

In [ ]:
import shutil
import os

final_folder = 'external_word_sample_renamed'
os.makedirs(final_folder, exist_ok=True)

source_folder = '/content/word_dataset/Cropped_word_dataset/word_dataset'
new_entries = []

for i, (filename, text) in enumerate(sampled, 1):
    src = os.path.join(source_folder, filename)
    if os.path.exists(src):
        ext = os.path.splitext(filename)[1]
        new_name = f'scene_word_{i:04d}{ext}'
        shutil.copy(src, os.path.join(final_folder, new_name))
        new_entries.append({
            'image': f'images/scene_word/{new_name}',
            'text': text,
            'source': 'natural_scene_dataset'
        })

print(f'Rebuilt: {len(new_entries)} entries')

In [ ]:
import pandas as pd

df_existing = pd.read_csv('/content/drive/MyDrive/Urdu_OCR/labels.csv')
print('Existing rows:', len(df_existing))

df_new = pd.DataFrame(new_entries)
print('New rows:', len(df_new))

df_combined = pd.concat([df_existing, df_new], ignore_index=True)
print('Combined total:', len(df_combined))

df_combined.to_csv('/content/drive/MyDrive/Urdu_OCR/labels_expanded.csv', index=False)
print('Saved: labels_expanded.csv')
print(df_combined['source'].value_counts())

In [ ]:
import cv2
import numpy as np
import os

def preprocess_standard(image_path, save_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    contrast_enhanced = clahe.apply(gray)
    target_w = 512
    max_h = 256
    h, w = contrast_enhanced.shape
    scale = target_w / w
    new_h = min(int(h * scale), max_h)
    resized = cv2.resize(contrast_enhanced, (target_w, new_h), interpolation=cv2.INTER_LANCZOS4)
    canvas = np.full((max_h, target_w), 255, dtype=np.uint8)
    canvas[:new_h, :] = resized
    denoised = cv2.fastNlMeansDenoising(canvas, h=5)
    binary = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 9, 4)
    cv2.imwrite(save_path, binary)
    return binary

os.makedirs('data/processed/scene_word', exist_ok=True)

processed_count = 0
for entry in new_entries:
    filename = os.path.basename(entry['image'])
    src = f'external_word_sample_renamed/{filename}'
    save_path = f'data/processed/scene_word/{filename}'
    result = preprocess_standard(src, save_path)
    if result is not None:
        processed_count += 1

print(f'Preprocessed {processed_count} new images')

In [ ]:
import matplotlib.pyplot as plt
import cv2

sample_indices = [0, 100, 300, 500, 999]

fig, axes = plt.subplots(len(sample_indices), 1, figsize=(8, 4*len(sample_indices)))
for i, idx in enumerate(sample_indices):
    entry = new_entries[idx]
    filename = os.path.basename(entry['image'])
    path = f'data/processed/scene_word/{filename}'
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'{filename} (ground truth: "{entry["text"]}")', fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('scene_word_processed_check.png', dpi=100, bbox_inches='tight')
plt.show()

from google.colab import files
files.download('scene_word_processed_check.png')

Use gentler preprocessing for this specific dataset

In [ ]:
clean_entries = [e for e in new_entries if '\ufffd' not in e['text'] and '□' not in e['text']]
print(f'Entries after removing corrupted labels: {len(clean_entries)} (was {len(new_entries)})')

In [ ]:
import cv2
import numpy as np
import os

def preprocess_small_scene_word(image_path, save_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Upscale first, with high-quality interpolation — preserves what little detail exists
    h, w = gray.shape
    scale = 512 / w
    new_h = int(h * scale)
    upscaled = cv2.resize(gray, (512, new_h), interpolation=cv2.INTER_CUBIC)

    # Gentle contrast boost only — no aggressive denoise/threshold that destroys small text
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(4, 4))
    enhanced = clahe.apply(upscaled)

    cv2.imwrite(save_path, enhanced)
    return enhanced

# Clear old (destroyed) processed versions and redo properly
import shutil
if os.path.exists('data/processed/scene_word'):
    shutil.rmtree('data/processed/scene_word')
os.makedirs('data/processed/scene_word', exist_ok=True)

processed_count = 0
for entry in clean_entries:
    filename = os.path.basename(entry['image'])
    src = f'external_word_sample_renamed/{filename}'
    save_path = f'data/processed/scene_word/{filename}'
    result = preprocess_small_scene_word(src, save_path)
    if result is not None:
        processed_count += 1

print(f'Reprocessed {processed_count} images with gentler pipeline')

In [ ]:
import matplotlib.pyplot as plt

sample_indices = [0, 100, 300, 500, 999] if len(clean_entries) > 999 else [0, 5, 10, 15, 20]

fig, axes = plt.subplots(len(sample_indices), 1, figsize=(8, 4*len(sample_indices)))
for i, idx in enumerate(sample_indices):
    entry = clean_entries[idx]
    filename = os.path.basename(entry['image'])
    path = f'data/processed/scene_word/{filename}'
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(filename, fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('scene_word_v2_check.png', dpi=100, bbox_inches='tight')
plt.show()

from google.colab import files
files.download('scene_word_v2_check.png')

In [ ]:
def preprocess_small_scene_word_v2(image_path, save_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Upscale with high-quality interpolation
    h, w = gray.shape
    scale = 512 / w
    new_h = int(h * scale)
    upscaled = cv2.resize(gray, (512, new_h), interpolation=cv2.INTER_CUBIC)

    # Gentle contrast boost
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(4, 4))
    enhanced = clahe.apply(upscaled)

    # Sharpening filter — counteracts the blur introduced by upscaling
    kernel = np.array([[-1,-1,-1], [-1, 9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(enhanced, -1, kernel)

    # Very light denoise AFTER sharpening, to clean up any noise the sharpening amplified
    denoised = cv2.fastNlMeansDenoising(sharpened, h=3)

    cv2.imwrite(save_path, denoised)
    return denoised

# Reprocess
import shutil
if os.path.exists('data/processed/scene_word'):
    shutil.rmtree('data/processed/scene_word')
os.makedirs('data/processed/scene_word', exist_ok=True)

processed_count = 0
for entry in clean_entries:
    filename = os.path.basename(entry['image'])
    src = f'external_word_sample_renamed/{filename}'
    save_path = f'data/processed/scene_word/{filename}'
    result = preprocess_small_scene_word_v2(src, save_path)
    if result is not None:
        processed_count += 1

print(f'Reprocessed {processed_count} images with sharpening')

In [ ]:
img = cv2.imread('data/processed/scene_word/scene_word_0301.jpg', cv2.IMREAD_GRAYSCALE)
plt.figure(figsize=(10, 6))
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.title('scene_word_0301.jpg - sharpened')
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.listdir('/content/drive/MyDrive/clean_urdu_dataset'))

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/clean_urdu_dataset/clean_urdu_dataset'))

In [ ]:
import os
path = '/content/drive/MyDrive/clean_urdu_dataset/clean_urdu_dataset/printed'
print(os.listdir(path))

In [ ]:
import pandas as pd
import os
import random
import shutil

# ===== STEP 1: Inspect CSV =====
csv_path = '/content/drive/MyDrive/clean_urdu_dataset/clean_urdu_dataset/printed/labels.csv'
df = pd.read_csv(csv_path)

print('=== CSV Info ===')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print()
print(df.head(5))
print()

# ===== STEP 2: Auto-detect image and text columns =====
possible_image_cols = ['image', 'filename', 'file', 'img', 'image_path', 'path']
possible_text_cols = ['text', 'label', 'ground_truth', 'gt', 'transcription']

image_col = next((c for c in df.columns if c.lower() in possible_image_cols), df.columns[0])
text_col = next((c for c in df.columns if c.lower() in possible_text_cols), df.columns[1])

print(f'Detected image column: "{image_col}"')
print(f'Detected text column: "{text_col}"')
print()

# ===== STEP 3: Check the images folder =====
images_folder = '/content/drive/MyDrive/clean_urdu_dataset/clean_urdu_dataset/printed/images'
print('Sample files in images folder:', os.listdir(images_folder)[:5])
print('Total files in images folder:', len(os.listdir(images_folder)))
print()

# ===== STEP 4: Sample 1500 entries =====
random.seed(42)
sample_size = min(1500, len(df))
df_sampled = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
print(f'Sampled {len(df_sampled)} entries')
print(df_sampled.head(5))
print()

# ===== STEP 5: Copy sampled images to a clean, renamed folder =====
dest_folder = '/content/printed_sample_renamed'
os.makedirs(dest_folder, exist_ok=True)

printed_entries = []
copied = 0
missing = []

for i, row in df_sampled.iterrows():
    orig_filename = str(row[image_col])
    src = os.path.join(images_folder, orig_filename)

    if os.path.exists(src):
        ext = os.path.splitext(orig_filename)[1]
        new_name = f'printed_{i+1:04d}{ext}'
        shutil.copy(src, os.path.join(dest_folder, new_name))
        printed_entries.append({
            'image': f'images/printed_word/{new_name}',
            'text': row[text_col],
            'source': 'clean_printed_dataset'
        })
        copied += 1
    else:
        missing.append(orig_filename)

print(f'Copied: {copied}')
print(f'Missing: {len(missing)}')
if missing:
    print('First few missing:', missing[:5])

In [ ]:
import os

base_path = '/content/drive/MyDrive/clean_urdu_dataset/clean_urdu_dataset/printed'
print('Contents of printed folder:')
print(os.listdir(base_path))
print()

# Check if there's a different subfolder structure
for item in os.listdir(base_path):
    full_path = os.path.join(base_path, item)
    if os.path.isdir(full_path):
        print(f'\nContents of {item}/:')
        print(os.listdir(full_path)[:10])

In [ ]:
import os

images_path = '/content/drive/MyDrive/clean_urdu_dataset/clean_urdu_dataset/printed/images'

# Try a fresh listing
print('Direct listdir:', len(os.listdir(images_path)))

# Try checking if a specific expected file exists directly (bypasses listdir caching issues)
test_file = os.path.join(images_path, 'printed_000001.png')
print('Does printed_000001.png exist directly?', os.path.exists(test_file))

# Check folder size/stats
print('Is it actually a directory?', os.path.isdir(images_path))

In [ ]:
import zipfile
import os

# Check if the zip is already uploaded in this session
if not os.path.exists('clean_urdu_dataset.zip'):
    from google.colab import files
    print('Please re-upload clean_urdu_dataset.zip:')
    uploaded = files.upload()

# Inspect structure
with zipfile.ZipFile('clean_urdu_dataset.zip', 'r') as z:
    names = z.namelist()
    print('Total entries in zip:', len(names))
    print('First 10 entries:')
    for n in names[:10]:
        print(' ', n)

In [ ]:
import pandas as pd
import random
import os

# Extract just the CSV first
with zipfile.ZipFile('clean_urdu_dataset.zip', 'r') as z:
    z.extract('clean_urdu_dataset/clean_urdu_dataset/printed/labels.csv', '/content/')

csv_path = '/content/clean_urdu_dataset/clean_urdu_dataset/printed/labels.csv'
df = pd.read_csv(csv_path)
print('CSV loaded:', df.shape)

# Rebuild the same 1500 sample (same random_state=42, so it's identical to before)
random.seed(42)
sample_size = min(1500, len(df))
df_sampled = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
print(f'Sampled {len(df_sampled)} entries')

# Extract only the 1500 needed images from the zip
dest_folder = '/content/printed_sample_renamed'
os.makedirs(dest_folder, exist_ok=True)

printed_entries = []
extracted = 0
missing = []

with zipfile.ZipFile('clean_urdu_dataset.zip', 'r') as z:
    for i, row in df_sampled.iterrows():
        orig_filename = str(row['image_name'])
        zip_internal_path = f'clean_urdu_dataset/clean_urdu_dataset/printed/images/{orig_filename}'

        if zip_internal_path in z.namelist():
            with z.open(zip_internal_path) as source_file:
                ext = os.path.splitext(orig_filename)[1]
                new_name = f'printed_{i+1:04d}{ext}'
                dest_path = os.path.join(dest_folder, new_name)
                with open(dest_path, 'wb') as dest_file:
                    dest_file.write(source_file.read())
                printed_entries.append({
                    'image': f'images/printed_word/{new_name}',
                    'text': row['label'],
                    'source': 'clean_printed_dataset'
                })
                extracted += 1
        else:
            missing.append(orig_filename)

print(f'\nExtracted: {extracted}')
print(f'Missing: {len(missing)}')

In [ ]:
print('Total entries in printed_entries:', len(printed_entries))
print('\nFirst 3 entries:')
for e in printed_entries[:3]:
    print(e)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

fig, axes = plt.subplots(5, 1, figsize=(10, 20))
for i in range(5):
    entry = printed_entries[i]
    filename = os.path.basename(entry['image'])
    img_path = os.path.join('printed_sample_renamed', filename)
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(filename, fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('printed_check.png', dpi=100, bbox_inches='tight')
plt.show()

from google.colab import files
files.download('printed_check.png')

In [ ]:
import pandas as pd

df_printed_check = pd.DataFrame(printed_entries)
print('Total rows:', len(df_printed_check))
print('Unique images:', df_printed_check['image'].nunique())
print('Any blank text:', df_printed_check['text'].isna().sum() + (df_printed_check['text']=='').sum())
print()
print('Source value counts:')
print(df_printed_check['source'].value_counts())
print()
print('First 5 rows:')
print(df_printed_check.head())

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

fig, axes = plt.subplots(3, 1, figsize=(10, 12))

for i in range(3):
    entry = printed_entries[i]
    filename = os.path.basename(entry['image'])
    img_path = os.path.join('printed_sample_renamed', filename)
    img = Image.open(img_path)

    axes[i].imshow(img)
    axes[i].set_title(f'{filename}', fontsize=10)
    axes[i].axis('off')

    # Print the exact text below, since matplotlib titles can garble RTL display
    print(f'{filename}:')
    print(f'  Text: {entry["text"]}')
    print()

plt.tight_layout()
plt.savefig('printed_text_match_check.png', dpi=100, bbox_inches='tight')
plt.show()

from google.colab import files
files.download('printed_text_match_check.png')

In [ ]:
import cv2
import numpy as np
import os

def preprocess_standard(image_path, save_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    contrast_enhanced = clahe.apply(gray)
    target_w = 512
    max_h = 256
    h, w = contrast_enhanced.shape
    scale = target_w / w
    new_h = min(int(h * scale), max_h)
    resized = cv2.resize(contrast_enhanced, (target_w, new_h), interpolation=cv2.INTER_LANCZOS4)
    canvas = np.full((max_h, target_w), 255, dtype=np.uint8)
    canvas[:new_h, :] = resized
    denoised = cv2.fastNlMeansDenoising(canvas, h=5)
    binary = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 9, 4)
    cv2.imwrite(save_path, binary)
    return binary

os.makedirs('data/processed/printed_word', exist_ok=True)

processed_count = 0
for entry in printed_entries:
    filename = os.path.basename(entry['image'])
    src = f'printed_sample_renamed/{filename}'
    save_path = f'data/processed/printed_word/{filename}'
    result = preprocess_standard(src, save_path)
    if result is not None:
        processed_count += 1

print(f'Preprocessed {processed_count} printed images')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.listdir('/content/drive/MyDrive/Urdu_OCR'))

In [ ]:
import pandas as pd

df_existing = pd.read_csv('/content/drive/MyDrive/Urdu_OCR/labels.csv')
print('Existing rows:', len(df_existing))

df_printed = pd.DataFrame(printed_entries)
print('New printed rows:', len(df_printed))

df_combined = pd.concat([df_existing, df_printed], ignore_index=True)
print('Combined total:', len(df_combined))

df_combined.to_csv('/content/drive/MyDrive/Urdu_OCR/labels_expanded.csv', index=False)
print('Saved: labels_expanded.csv')
print(df_combined['source'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

sample_indices = [0, 300, 700, 1100, 1499]

fig, axes = plt.subplots(len(sample_indices), 1, figsize=(10, 4*len(sample_indices)))
for i, idx in enumerate(sample_indices):
    entry = printed_entries[idx]
    filename = os.path.basename(entry['image'])
    path = f'data/processed/printed_word/{filename}'
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(filename, fontsize=10)
    axes[i].axis('off')
    print(f'{filename}: "{entry["text"]}"')

plt.tight_layout()
plt.savefig('printed_processed_check.png', dpi=100, bbox_inches='tight')
plt.show()

from google.colab import files
files.download('printed_processed_check.png')

In [ ]:
import shutil
import os

os.makedirs('/content/drive/MyDrive/external_datasets', exist_ok=True)

shutil.make_archive('printed_raw_1500', 'zip', 'printed_sample_renamed')
shutil.copy('printed_raw_1500.zip', '/content/drive/MyDrive/external_datasets/printed_raw_1500.zip')
print('Saved raw printed images to Drive')

shutil.make_archive('printed_processed_1500', 'zip', 'data/processed/printed_word')
shutil.copy('printed_processed_1500.zip', '/content/drive/MyDrive/external_datasets/printed_processed_1500.zip')
print('Saved processed printed images to Drive')

print('\nFiles in external_datasets:')
print(os.listdir('/content/drive/MyDrive/external_datasets'))

In [ ]:
import random
import os
import shutil
import cv2
import numpy as np

# 1. Re-extract the rar
!apt-get install -y unrar -q
!unrar x "/content/drive/MyDrive/external_datasets/Cropped_word_dataset.rar" "/content/word_dataset/" -y

# 2. Re-parse ground truth
gt_path = '/content/word_dataset/Cropped_word_dataset/gt_file.tsv'
entries = []
with open(gt_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            filename, text = parts
            if len(text.strip()) >= 2:
                entries.append((filename, text))

print(f'Total valid entries: {len(entries)}')

# 3. Re-sample same 1000 (same seed = identical to before)
random.seed(42)
sample_size = min(1000, len(entries))
sampled = random.sample(entries, sample_size)
print(f'Sampled: {len(sampled)}')

# 4. Filter out corrupted labels
clean_entries_raw = [(f, t) for f, t in sampled if '\ufffd' not in t and '□' not in t]
print(f'After filtering corrupted labels: {len(clean_entries_raw)}')

# 5. Copy and rename images
source_folder = '/content/word_dataset/Cropped_word_dataset/word_dataset'
dest_folder = '/content/external_word_sample_renamed'
os.makedirs(dest_folder, exist_ok=True)

new_entries = []
for i, (filename, text) in enumerate(clean_entries_raw, 1):
    src = os.path.join(source_folder, filename)
    if os.path.exists(src):
        ext = os.path.splitext(filename)[1]
        new_name = f'scene_word_{i:04d}{ext}'
        shutil.copy(src, os.path.join(dest_folder, new_name))
        new_entries.append({
            'image': f'images/scene_word/{new_name}',
            'text': text,
            'source': 'natural_scene_dataset'
        })

print(f'Copied and renamed: {len(new_entries)}')

# 6. Preprocess with the gentle pipeline (final version that worked)
def preprocess_small_scene_word_final(image_path, save_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    scale = 512 / w
    new_h = int(h * scale)
    upscaled = cv2.resize(gray, (512, new_h), interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(4, 4))
    enhanced = clahe.apply(upscaled)
    cv2.imwrite(save_path, enhanced)
    return enhanced

os.makedirs('data/processed/scene_word', exist_ok=True)
processed_count = 0
for entry in new_entries:
    filename = os.path.basename(entry['image'])
    src = f'external_word_sample_renamed/{filename}'
    save_path = f'data/processed/scene_word/{filename}'
    result = preprocess_small_scene_word_final(src, save_path)
    if result is not None:
        processed_count += 1

print(f'Preprocessed: {processed_count}')

In [ ]:
import shutil
import os

os.makedirs('/content/drive/MyDrive/external_datasets', exist_ok=True)

# Save raw (unprocessed) scene word images
shutil.make_archive('scene_word_raw_1000', 'zip', 'external_word_sample_renamed')
shutil.copy('scene_word_raw_1000.zip', '/content/drive/MyDrive/external_datasets/scene_word_raw_1000.zip')
print('Saved raw scene word images to Drive')

# Save processed scene word images
shutil.make_archive('scene_word_processed_1000', 'zip', 'data/processed/scene_word')
shutil.copy('scene_word_processed_1000.zip', '/content/drive/MyDrive/external_datasets/scene_word_processed_1000.zip')
print('Saved processed scene word images to Drive')

# Verify
print('\nFiles in external_datasets:')
print(os.listdir('/content/drive/MyDrive/external_datasets'))

In [ ]:
import os
import shutil
import zipfile

project_root = '/content/drive/MyDrive/Urdu_OCR'
external = '/content/drive/MyDrive/external_datasets'

# 1. Create matching subfolders inside the main project
for folder in ['printed', 'scene_word']:
    os.makedirs(f'{project_root}/data/processed/{folder}', exist_ok=True)
    os.makedirs(f'{project_root}/data/raw/{folder}', exist_ok=True)

# 2. Unzip processed images into project's data/processed/
with zipfile.ZipFile(f'{external}/printed_processed_1500.zip', 'r') as z:
    z.extractall(f'{project_root}/data/processed/printed')

with zipfile.ZipFile(f'{external}/scene_word_processed_1000.zip', 'r') as z:
    z.extractall(f'{project_root}/data/processed/scene_word')

# 3. Unzip raw images into project's data/raw/ (backup/reference)
with zipfile.ZipFile(f'{external}/printed_raw_1500.zip', 'r') as z:
    z.extractall(f'{project_root}/data/raw/printed')

with zipfile.ZipFile(f'{external}/scene_word_raw_1000.zip', 'r') as z:
    z.extractall(f'{project_root}/data/raw/scene_word')

# 4. Verify counts
for folder in ['printed', 'scene_word']:
    count = len(os.listdir(f'{project_root}/data/processed/{folder}'))
    print(f'{folder}: {count} processed images now in Urdu_OCR project')

In [ ]:
# 1. Check if printed labels were ever saved as a file
print(os.listdir('/content/drive/MyDrive/external_datasets'))

# 2. Check for any leftover extraction folder with printed ground truth
print(os.listdir('/content'))

In [ ]:
print(os.listdir('/content/clean_urdu_dataset'))
print(os.listdir('/content/printed_sample_renamed')[:10])  # just first 10 filenames

In [ ]:
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'

print("=== Original Week 1-2 dataset ===")
for folder in ['synthetic', 'handwritten', 'newspaper', 'signboard']:
    path = f'{project_root}/data/processed/{folder}'
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f'{folder}: {count}')
    else:
        print(f'{folder}: folder not found')

print("\n=== External datasets (merged into project) ===")
for folder in ['printed', 'scene_word']:
    path = f'{project_root}/data/processed/{folder}'
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f'{folder}: {count}')
    else:
        print(f'{folder}: folder not found')

print("\n=== labels.csv check ===")
labels_path = f'{project_root}/data/labels.csv'
if os.path.exists(labels_path):
    import pandas as pd
    df = pd.read_csv(labels_path)
    print(f'labels.csv exists with {len(df)} entries')
    if 'source' in df.columns:
        print(df['source'].value_counts())
else:
    print('labels.csv does NOT exist yet — this is what we still need to build')

In [ ]:
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'
print("Top level:", os.listdir(project_root))

# Check common variations
for sub in ['data', 'Data']:
    p = f'{project_root}/{sub}'
    if os.path.exists(p):
        print(f"\nInside {sub}/:", os.listdir(p))
        for sub2 in ['processed', 'Processed', 'raw', 'Raw']:
            p2 = f'{p}/{sub2}'
            if os.path.exists(p2):
                print(f"Inside {sub}/{sub2}/:", os.listdir(p2))

In [ ]:
import os
import pandas as pd

project_root = '/content/drive/MyDrive/Urdu_OCR'

# Check raw folder counts
print("=== Raw image counts ===")
print("raw/printed:", len(os.listdir(f'{project_root}/data/raw/printed')))
print("raw/scene_word:", len(os.listdir(f'{project_root}/data/raw/scene_word')))
print("processed/printed:", len(os.listdir(f'{project_root}/data/processed/printed')))
print("processed/scene_word:", len(os.listdir(f'{project_root}/data/processed/scene_word')))

# Check existing labels.csv
print("\n=== labels.csv ===")
df1 = pd.read_csv(f'{project_root}/labels.csv')
print(f"Rows: {len(df1)}")
print(f"Columns: {list(df1.columns)}")
print(df1.head(3))

# Check existing labels_expanded.csv
print("\n=== labels_expanded.csv ===")
df2 = pd.read_csv(f'{project_root}/labels_expanded.csv')
print(f"Rows: {len(df2)}")
print(f"Columns: {list(df2.columns)}")
print(df2.head(3))
if 'source' in df2.columns:
    print(df2['source'].value_counts())

In [ ]:
import pandas as pd
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'
df2 = pd.read_csv(f'{project_root}/labels_expanded.csv')

# Check path format for printed rows
printed_rows = df2[df2['source'] == 'clean_printed_dataset']
print("Sample printed image paths:")
print(printed_rows['image'].head(5).tolist())

# Check actual filenames on disk for scene_word
print("\nSample scene_word filenames on disk (processed):")
print(sorted(os.listdir(f'{project_root}/data/processed/scene_word'))[:5])
print("\nSample scene_word filenames on disk (raw):")
print(sorted(os.listdir(f'{project_root}/data/raw/scene_word'))[:5])

In [ ]:
import random
import os
import pandas as pd

project_root = '/content/drive/MyDrive/Urdu_OCR'

# 1. Re-extract ground truth (rar already extracted earlier at /content/word_dataset;
#    re-run extraction only if that folder is gone this session)
if not os.path.exists('/content/word_dataset/Cropped_word_dataset/gt_file.tsv'):
    !apt-get install -y unrar -q
    !unrar x "/content/drive/MyDrive/external_datasets/Cropped_word_dataset.rar" "/content/word_dataset/" -y

gt_path = '/content/word_dataset/Cropped_word_dataset/gt_file.tsv'
entries = []
with open(gt_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            filename, text = parts
            if len(text.strip()) >= 2:
                entries.append((filename, text))

print(f'Total valid entries: {len(entries)}')

# 2. Re-sample with same seed = same 1000, same order as before
random.seed(42)
sample_size = min(1000, len(entries))
sampled = random.sample(entries, sample_size)

# 3. Filter corrupted labels (same as before — expect 0 filtered this run)
clean_entries_raw = [(f, t) for f, t in sampled if '\ufffd' not in t and '□' not in t]
print(f'After filtering: {len(clean_entries_raw)}')

# 4. Build rows matching the filenames already on disk (scene_word_0001.jpg, ...)
scene_word_rows = []
for i, (orig_filename, text) in enumerate(clean_entries_raw, 1):
    ext = os.path.splitext(orig_filename)[1]
    new_name = f'scene_word_{i:04d}{ext}'
    scene_word_rows.append({
        'image': f'images/scene_word/{new_name}',
        'text': text,
        'source': 'natural_scene_dataset'
    })

print(f'Scene word rows built: {len(scene_word_rows)}')

# 5. Sanity check — confirm filenames match what's on disk
disk_files = set(os.listdir(f'{project_root}/data/processed/scene_word'))
csv_files = set(row['image'].split('/')[-1] for row in scene_word_rows)
print(f'Match check: {len(csv_files & disk_files)} / {len(disk_files)} filenames match')

# 6. Append to labels_expanded.csv and save
df = pd.read_csv(f'{project_root}/labels_expanded.csv')
new_df = pd.DataFrame(scene_word_rows)
df_final = pd.concat([df, new_df], ignore_index=True)

df_final.to_csv(f'{project_root}/labels_expanded.csv', index=False)

print(f'\nFinal labels_expanded.csv: {len(df_final)} rows')
print(df_final['source'].value_counts())

In [ ]:
import shutil
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'

# Zip the entire processed folder (all 6 categories: synthetic, handwritten,
# newspaper, signboard, printed, scene_word) into one backup archive
shutil.make_archive(
    f'{project_root}/backup_processed_2633_images',
    'zip',
    f'{project_root}/data/processed'
)
print('Saved processed images backup to Drive')

# Copy the final labels file with a clear versioned name for safekeeping
shutil.copy(
    f'{project_root}/labels_expanded.csv',
    f'{project_root}/labels_final_2633.csv'
)
print('Saved labels_final_2633.csv to Drive')

# Verify what's now in the project root
print('\nFiles in Urdu_OCR:')
print(os.listdir(project_root))

In [ ]:
from google.colab import files

# Download the master labels CSV
files.download(f'{project_root}/labels_final_2633.csv')

# Download the zipped processed images (all 2633 images, ~could be large —
# this may take a bit depending on your connection)
files.download(f'{project_root}/backup_processed_2633_images.zip')

In [ ]:
import shutil
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'

# 1. Create a staging folder with everything we want in the final archive
staging = '/content/full_backup_staging'
os.makedirs(staging, exist_ok=True)

# Copy raw images (all categories)
shutil.copytree(f'{project_root}/data/raw', f'{staging}/data/raw', dirs_exist_ok=True)
print('Copied raw images to staging')

# Copy processed images (all categories)
shutil.copytree(f'{project_root}/data/processed', f'{staging}/data/processed', dirs_exist_ok=True)
print('Copied processed images to staging')

# Copy all label CSVs
for csv_name in ['labels.csv', 'labels_expanded.csv', 'labels_final_2633.csv']:
    src = f'{project_root}/{csv_name}'
    if os.path.exists(src):
        shutil.copy(src, f'{staging}/{csv_name}')
        print(f'Copied {csv_name} to staging')

# 2. Zip the entire staging folder into one archive
archive_name = 'Urdu_OCR_complete_backup'
shutil.make_archive(archive_name, 'zip', staging)
print(f'\nCreated {archive_name}.zip')

# 3. Save the complete archive to Drive
shutil.copy(f'{archive_name}.zip', f'{project_root}/{archive_name}.zip')
print('Saved complete backup to Drive')

# 4. Check final size
size_mb = os.path.getsize(f'{archive_name}.zip') / (1024 * 1024)
print(f'\nArchive size: {size_mb:.1f} MB')

In [ ]:
from google.colab import files
files.download(f'{archive_name}.zip')

In [ ]:
import zipfile

project_root = '/content/drive/MyDrive/Urdu_OCR'

print("=== Inside urdu_ocr_dataset.zip ===")
with zipfile.ZipFile(f'{project_root}/urdu_ocr_dataset.zip', 'r') as z:
    names = z.namelist()
    print(f'Total files: {len(names)}')
    print(names[:10])

print("\n=== Inside processed_images_final.zip ===")
with zipfile.ZipFile(f'{project_root}/processed_images_final.zip', 'r') as z:
    names = z.namelist()
    print(f'Total files: {len(names)}')
    print(names[:10])

In [ ]:
import shutil
import os
import zipfile

project_root = '/content/drive/MyDrive/Urdu_OCR'
staging = '/content/full_backup_staging'

# Clean slate
if os.path.exists(staging):
    shutil.rmtree(staging)
os.makedirs(f'{staging}/data/raw', exist_ok=True)
os.makedirs(f'{staging}/data/processed', exist_ok=True)

# 1. Extract Week 1-2 RAW images + labels.csv from urdu_ocr_dataset.zip
with zipfile.ZipFile(f'{project_root}/urdu_ocr_dataset.zip', 'r') as z:
    z.extractall('/content/week1_2_raw_temp')

# Move the 4 category folders (from images/) into staging/data/raw
for cat in ['synthetic', 'handwritten', 'newspaper', 'signboard']:
    src = f'/content/week1_2_raw_temp/images/{cat}'
    dst = f'{staging}/data/raw/{cat}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Copied raw/{cat}: {len(os.listdir(dst))} files')

# 2. Extract Week 1-2 PROCESSED images from processed_images_final.zip
with zipfile.ZipFile(f'{project_root}/processed_images_final.zip', 'r') as z:
    z.extractall('/content/week1_2_processed_temp')

for cat in ['synthetic', 'handwritten', 'newspaper', 'signboard']:
    src = f'/content/week1_2_processed_temp/{cat}'
    dst = f'{staging}/data/processed/{cat}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Copied processed/{cat}: {len(os.listdir(dst))} files')

# 3. Copy printed + scene_word (raw and processed) — already in place
for cat in ['printed', 'scene_word']:
    shutil.copytree(f'{project_root}/data/raw/{cat}', f'{staging}/data/raw/{cat}', dirs_exist_ok=True)
    shutil.copytree(f'{project_root}/data/processed/{cat}', f'{staging}/data/processed/{cat}', dirs_exist_ok=True)
    print(f'Copied raw/{cat} and processed/{cat}')

# 4. Copy all label CSVs
for csv_name in ['labels.csv', 'labels_expanded.csv', 'labels_final_2633.csv']:
    src = f'{project_root}/{csv_name}'
    if os.path.exists(src):
        shutil.copy(src, f'{staging}/{csv_name}')
        print(f'Copied {csv_name}')

# 5. Verify total counts before zipping
print('\n=== Final staging counts ===')
for split in ['raw', 'processed']:
    print(f'\n{split}:')
    total = 0
    for cat in ['synthetic', 'handwritten', 'newspaper', 'signboard', 'printed', 'scene_word']:
        path = f'{staging}/data/{split}/{cat}'
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        total += count
        print(f'  {cat}: {count}')
    print(f'  TOTAL {split}: {total}')

# 6. Zip everything
archive_name = 'Urdu_OCR_complete_backup'
shutil.make_archive(archive_name, 'zip', staging)
shutil.copy(f'{archive_name}.zip', f'{project_root}/{archive_name}.zip')

size_mb = os.path.getsize(f'{archive_name}.zip') / (1024 * 1024)
print(f'\nArchive saved to Drive. Size: {size_mb:.1f} MB')

In [ ]:
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'
print(os.listdir(project_root))

In [ ]:
from google.colab import files
files.download('Urdu_OCR_complete_backup.zip')

step 2

In [ ]:
!pip install transformers torch pillow pandas -q

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, image_root, processor):
        self.data = pd.read_csv(csv_path)
        self.image_root = image_root
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = os.path.join(self.image_root, row['image'])
        image = Image.open(image_path).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

# Load the TrOCR processor
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

# Create dataset — pointed at your final master CSV
dataset = UrduOCRDataset(
    csv_path=f'{project_root}/labels_final_2633.csv',
    image_root=f'{project_root}/data/processed',
    processor=processor
)

# Test it loads correctly
sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

# Create train / test split (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)
print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

In [ ]:
!pip install sentencepiece -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers torch pillow pandas sentencepiece -q

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, image_root, processor):
        self.data = pd.read_csv(csv_path)
        self.image_root = image_root
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = os.path.join(self.image_root, row['image'])
        image = Image.open(image_path).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

dataset = UrduOCRDataset(
    csv_path=f'{project_root}/labels_final_2633.csv',
    image_root=f'{project_root}/data/processed',
    processor=processor
)

sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

In [ ]:
!pip install sentencepiece --force-reinstall -q
import sentencepiece
print("sentencepiece version:", sentencepiece.__version__)

In [ ]:
!rm -rf /root/.cache/huggingface
from transformers import TrOCRProcessor
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
print("Processor loaded successfully!")

In [ ]:
!pip uninstall transformers -y -q
!pip install transformers==4.41.2 -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install transformers==4.41.2 torch pillow pandas sentencepiece -q

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, image_root, processor):
        self.data = pd.read_csv(csv_path)
        self.image_root = image_root
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = os.path.join(self.image_root, row['image'])
        image = Image.open(image_path).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

dataset = UrduOCRDataset(
    csv_path=f'{project_root}/labels_final_2633.csv',
    image_root=f'{project_root}/data/processed',
    processor=processor
)

sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Dataset loaded: 2633 samples


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Urdu_OCR/data/processed/images/synthetic/synthetic_001.png'

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, image_root, processor):
        self.data = pd.read_csv(csv_path)
        self.image_root = image_root
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        # Strip leading "images/" if present, since actual files don't have that folder
        rel_path = row['image']
        if rel_path.startswith('images/'):
            rel_path = rel_path[len('images/'):]
        image_path = os.path.join(self.image_root, rel_path)
        image = Image.open(image_path).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

dataset = UrduOCRDataset(
    csv_path=f'{project_root}/labels_final_2633.csv',
    image_root=f'{project_root}/data/processed',
    processor=processor
)

sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Dataset loaded: 2633 samples


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Urdu_OCR/data/processed/synthetic/synthetic_001.png'

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/Urdu_OCR/data/processed/synthetic'))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Urdu_OCR/data/processed/synthetic'

In [ ]:
import zipfile
import shutil
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'

with zipfile.ZipFile(f'{project_root}/processed_images_final.zip', 'r') as z:
    z.extractall('/content/week1_2_processed_temp')

for cat in ['synthetic', 'handwritten', 'newspaper', 'signboard']:
    src = f'/content/week1_2_processed_temp/{cat}'
    dst = f'{project_root}/data/processed/{cat}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Created data/processed/{cat}: {len(os.listdir(dst))} files')

print('\nFinal data/processed/ contents:', os.listdir(f'{project_root}/data/processed'))

Created data/processed/synthetic: 24 files
Created data/processed/handwritten: 78 files
Created data/processed/newspaper: 17 files
Created data/processed/signboard: 14 files

Final data/processed/ contents: ['printed', 'scene_word', 'synthetic', 'handwritten', 'newspaper', 'signboard']


In [ ]:
import zipfile
import shutil
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'

with zipfile.ZipFile(f'{project_root}/urdu_ocr_dataset.zip', 'r') as z:
    z.extractall('/content/week1_2_raw_temp')

for cat in ['synthetic', 'handwritten', 'newspaper', 'signboard']:
    src = f'/content/week1_2_raw_temp/images/{cat}'
    dst = f'{project_root}/data/raw/{cat}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Created data/raw/{cat}: {len(os.listdir(dst))} files')

print('\nFinal data/raw/ contents:', os.listdir(f'{project_root}/data/raw'))

Created data/raw/synthetic: 24 files
Created data/raw/handwritten: 78 files
Created data/raw/newspaper: 17 files
Created data/raw/signboard: 14 files

Final data/raw/ contents: ['printed', 'scene_word', 'synthetic', 'handwritten', 'newspaper', 'signboard']


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd
import os

project_root = '/content/drive/MyDrive/Urdu_OCR'

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, image_root, processor):
        self.data = pd.read_csv(csv_path)
        self.image_root = image_root
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        rel_path = row['image']
        if rel_path.startswith('images/'):
            rel_path = rel_path[len('images/'):]
        image_path = os.path.join(self.image_root, rel_path)
        image = Image.open(image_path).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

dataset = UrduOCRDataset(
    csv_path=f'{project_root}/labels_final_2633.csv',
    image_root=f'{project_root}/data/processed',
    processor=processor
)

sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Dataset loaded: 2633 samples


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Urdu_OCR/data/processed/synthetic/synthetic_001.png'

In [ ]:
import os
print(sorted(os.listdir('/content/drive/MyDrive/Urdu_OCR/data/processed/synthetic')))

['synthetic_01.png', 'synthetic_02.png', 'synthetic_03.png', 'synthetic_04.png', 'synthetic_05.png', 'synthetic_06.png', 'synthetic_07.png', 'synthetic_08.png', 'synthetic_09.png', 'synthetic_10.png', 'synthetic_11.png', 'synthetic_12.png', 'synthetic_13.png', 'synthetic_14.png', 'synthetic_15.png', 'synthetic_16.png', 'synthetic_17.png', 'synthetic_18.png', 'synthetic_19.png', 'synthetic_20.png', 'synthetic_21.png', 'synthetic_22.png', 'synthetic_23.png', 'synthetic_24.png']


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd
import os
import re

project_root = '/content/drive/MyDrive/Urdu_OCR'

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, image_root, processor):
        self.data = pd.read_csv(csv_path)
        self.image_root = image_root
        self.processor = processor
        # Pre-build a lookup: {(category, number): actual_filename} for every folder
        self.file_lookup = {}
        for category in os.listdir(image_root):
            cat_path = os.path.join(image_root, category)
            if os.path.isdir(cat_path):
                for fname in os.listdir(cat_path):
                    match = re.search(r'(\d+)(?=\.\w+$)', fname)
                    if match:
                        num = int(match.group(1))
                        self.file_lookup[(category, num)] = fname
        print(f'Dataset loaded: {len(self.data)} samples')
        print(f'Indexed {len(self.file_lookup)} files across {len(os.listdir(image_root))} categories')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        rel_path = row['image']
        if rel_path.startswith('images/'):
            rel_path = rel_path[len('images/'):]
        parts = rel_path.split('/')
        category = parts[0]
        csv_filename = parts[-1]

        # Try exact match first
        image_path = os.path.join(self.image_root, category, csv_filename)
        if not os.path.exists(image_path):
            # Fall back to number-based lookup (handles padding differences)
            match = re.search(r'(\d+)(?=\.\w+$)', csv_filename)
            if match:
                num = int(match.group(1))
                actual_fname = self.file_lookup.get((category, num))
                if actual_fname:
                    image_path = os.path.join(self.image_root, category, actual_fname)

        image = Image.open(image_path).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

dataset = UrduOCRDataset(
    csv_path=f'{project_root}/labels_final_2633.csv',
    image_root=f'{project_root}/data/processed',
    processor=processor
)

sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Dataset loaded: 2633 samples
Indexed 2633 files across 6 categories
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 2106
Testing samples: 527


In [ ]:
# Save the working Step 2 code to your project for reference
code = '''
# UrduOCRDataset - Week 3 final working version
# Handles filename padding mismatches automatically via number-based lookup

import torch
from torch.utils.data import Dataset
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd
import os
import re

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, image_root, processor):
        self.data = pd.read_csv(csv_path)
        self.image_root = image_root
        self.processor = processor
        self.file_lookup = {}
        for category in os.listdir(image_root):
            cat_path = os.path.join(image_root, category)
            if os.path.isdir(cat_path):
                for fname in os.listdir(cat_path):
                    match = re.search(r"(\\d+)(?=\\.\\w+$)", fname)
                    if match:
                        num = int(match.group(1))
                        self.file_lookup[(category, num)] = fname

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        rel_path = row["image"]
        if rel_path.startswith("images/"):
            rel_path = rel_path[len("images/"):]
        parts = rel_path.split("/")
        category = parts[0]
        csv_filename = parts[-1]
        image_path = os.path.join(self.image_root, category, csv_filename)
        if not os.path.exists(image_path):
            match = re.search(r"(\\d+)(?=\\.\\w+$)", csv_filename)
            if match:
                num = int(match.group(1))
                actual_fname = self.file_lookup.get((category, num))
                if actual_fname:
                    image_path = os.path.join(self.image_root, category, actual_fname)
        image = Image.open(image_path).convert("RGB")
        encoding = self.processor(image, return_tensors="pt")
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(row["text"], padding="max_length", max_length=128).input_ids
        labels = torch.tensor(labels)
        return {"pixel_values": pixel_values, "labels": labels}
'''

with open('/content/drive/MyDrive/Urdu_OCR/urdu_ocr_dataset_class.py', 'w') as f:
    f.write(code)
print('Saved working Dataset class to Drive')

Saved working Dataset class to Drive


In [ ]:
import shutil
shutil.copy(
    '/content/drive/MyDrive/Urdu_OCR/labels_final_2633.csv',
    '/content/drive/MyDrive/Urdu_OCR/data/labels.csv'  # overwrite the old one if this is in your repo's synced folder
)

'/content/drive/MyDrive/Urdu_OCR/data/labels.csv'

In [ ]:
import pandas as pd
import os

path = '/content/drive/MyDrive/Urdu_OCR/data/labels.csv'

if os.path.exists(path):
    df = pd.read_csv(path)
    print(f'Rows: {len(df)}')
    print(f'Columns: {list(df.columns)}')
    if 'source' in df.columns:
        print(df['source'].value_counts())
else:
    print('File does not exist at this path yet')

Rows: 2633
Columns: ['image', 'text', 'source']
source
clean_printed_dataset    1500
natural_scene_dataset    1000
handwritten                78
synthetic                  24
newspaper                  17
signboard                  14
Name: count, dtype: int64


In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/Urdu_OCR/data/labels.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>